In [4]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import requests
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [5]:
%pip install cloudscraper
import cloudscraper

Note: you may need to restart the kernel to use updated packages.


In [6]:
# Character Tags
characters = {
    'Naruto': 'uzumaki_naruto',
    'Luffy': 'monkey_d._luffy',
    'Isagi': 'isagi_yoichi',
    'Toji': 'fushiguro_toji',         
    'Goku': 'son_goku',              
    'Zoro': 'roronoa_zoro',
    'Ichigo': 'kurosaki_ichigo',
    'Saitama': 'saitama_(one-punch_man)',
    'Gojo': 'gojo_satoru',
    'Denji': 'denji_(chainsaw_man)'    
}

DATASET_DIR = "manga_dataset"
LIMIT = 150

os.makedirs(DATASET_DIR, exist_ok=True)

scraper = cloudscraper.create_scraper(browser={
    'browser': 'chrome',
    'platform': 'windows',
    'desktop': True
})

In [4]:
# Added a Name Tag so Safebooru doesn't block me !
HEADERS = {'User-Agent': 'MangaClassifierProject/1.0 (Student AI Project)'}

for character, tag in characters.items():
    character_dir = os.path.join(DATASET_DIR, tag)
    os.makedirs(character_dir, exist_ok=True)


    req_url = f"https://safebooru.donmai.us/posts.json?tags={tag}&limit={LIMIT}"
    response = scraper.get(req_url,headers=HEADERS)

    if response.status_code !=200:
        print(f"Failed to fetch data for {character} (Status Code: {response.status_code})")
        continue
    
    data = response.json()
    downloaded = 0

    for post in data:
        if 'file_url' not in post:
            continue    

        image_url = post['file_url']
        extension = image_url.split('.')[-1]

        if extension.lower() not in ['jpg', 'jpeg', 'png','webp']:
            continue

        image_path = os.path.join(character_dir, f"{character}_{downloaded}.{extension}")

        try:
            img_response = scraper.get(image_url,headers=HEADERS, timeout=100)
            if img_response.status_code == 200 and 'image' in img_response.headers.get('Content-Type', ''):
                with open(image_path, 'wb') as handler:
                    handler.write(img_response.content)
                downloaded += 1
            else:
                print(f"Cloudflare blocked image {downloaded} for {character}")

        except Exception as e:  
            pass

    print(f"Downloaded {downloaded} images for {character}")
    time.sleep(2)  # Being polite and avoid hitting the server too hard

print("Data collection complete!")

Downloaded 148 images for Naruto
Downloaded 149 images for Luffy
Downloaded 148 images for Isagi
Downloaded 149 images for Toji
Downloaded 149 images for Goku
Downloaded 150 images for Zoro
Downloaded 149 images for Ichigo
Downloaded 148 images for Saitama
Downloaded 150 images for Gojo
Downloaded 148 images for Denji
Data collection complete!


In [7]:
import warnings
warnings.filterwarnings("ignore")

In [8]:
# data transformations for training and validation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [9]:
dataset = datasets.ImageFolder(root=DATASET_DIR, transform=transform)
data_loader = DataLoader(dataset, batch_size=32, shuffle=False)

class_names = dataset.classes
print(f"Classes found: {class_names}")
print(f"Total images: {len(dataset)}")

Classes found: ['denji_(chainsaw_man)', 'fushiguro_toji', 'gojo_satoru', 'isagi_yoichi', 'kurosaki_ichigo', 'monkey_d._luffy', 'roronoa_zoro', 'saitama_(one-punch_man)', 'son_goku', 'uzumaki_naruto']
Total images: 1488


In [11]:
# Loading Pretrained ResNet50 Model

resnet50 = models.resnet50(pretrained=True)
for param in resnet50.parameters():
    param.requires_grad = False  # Freeze the p

print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [12]:
list(resnet50.children())[-1]

Linear(in_features=2048, out_features=1000, bias=True)

In [15]:
feature_extractor = nn.Sequential(*list(resnet50.children())[:-1]) # chopping off final layer
print(feature_extractor)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, ker

In [ ]:
features_list = []
labels_list = []

with torch.no_grad():
    for images, labels in data_loader:
        features = feature_extractor(images)
        features = features.view(features.size(0), -1).numpy()

        
        # handle edge case when batch size is 1
        if features.ndim == 1:
            features = np.expand_dims(features, axis=0)
        
        features_list.append(features)
        labels_list.append(labels.numpy())


print(f"Extracted features shape: {features_list[0].shape}")

X = np.vstack(features_list)  # Stack all feature batches vertically
y = np.hstack(labels_list)
print(f"Final feature matrix shape: {X.shape}")
print(f"Final labels shape: {y.shape}")




In [ ]:
print(y[0])

NameError: name 'X' is not defined

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
pca = PCA(n_components= 0.90, random_state = 42)
